# detrflow — RT-DETR Training on COCO

**Model:** `PekingU/rtdetr_r50vd` (HuggingFace)  
**Dataset:** COCO 2017  
**Hardware:** Google Colab T4 / A100  
**Repo:** [github.com/umutonuryasar/detrflow](https://github.com/umutonuryasar/detrflow)

---
### Before you start
1. `Runtime → Change runtime type → GPU (T4 or A100)`
2. Run cells top to bottom
3. Checkpoints are saved to Google Drive automatically

## 0 — GPU check

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (!!)'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

assert torch.cuda.is_available(), "No GPU found! Go to Runtime > Change runtime type > GPU."

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/detrflow'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CKPT_DIR}")

## 2 — Clone repo & install dependencies

In [ ]:
import os

if not os.path.exists('/content/detrflow'):
    !git clone https://github.com/umutonuryasar/detrflow.git /content/detrflow
else:
    !git -C /content/detrflow pull
    print("Repo already exists — pulled latest.")

%cd /content/detrflow

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed.")

## 3 — Download COCO 2017

> **~25 GB total.** Takes ~10 min on T4. Skip this cell if you have already downloaded COCO.

In [ ]:
import os

COCO_DIR = '/content/detrflow/data/coco'
os.makedirs(COCO_DIR, exist_ok=True)

files = {
    'train2017.zip'                  : 'http://images.cocodataset.org/zips/train2017.zip',
    'val2017.zip'                    : 'http://images.cocodataset.org/zips/val2017.zip',
    'annotations_trainval2017.zip'   : 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
}

for fname, url in files.items():
    dest = f'{COCO_DIR}/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        !wget -q --show-progress {url} -O {dest}
    else:
        print(f'Already exists: {fname}')

# Unzip (skip if already extracted)
if not os.path.exists(f'{COCO_DIR}/train2017'):
    print('Extracting train2017...')
    !unzip -q {COCO_DIR}/train2017.zip -d {COCO_DIR}
if not os.path.exists(f'{COCO_DIR}/val2017'):
    print('Extracting val2017...')
    !unzip -q {COCO_DIR}/val2017.zip -d {COCO_DIR}
if not os.path.exists(f'{COCO_DIR}/annotations'):
    print('Extracting annotations...')
    !unzip -q {COCO_DIR}/annotations_trainval2017.zip -d {COCO_DIR}

print('COCO ready.')
!ls {COCO_DIR}

## 4 — Training config

In [ ]:
import yaml

with open('configs/rtdetr_r50_coco.yaml') as f:
    cfg = yaml.safe_load(f)

# Override output dir to point to Drive
cfg['output_dir'] = CKPT_DIR

# Colab T4 optimizations
cfg['batch_size']       = 8   # T4 16 GB → batch 8 safe with fp16
cfg['grad_accum_steps'] = 2   # effective batch = 16
cfg['num_epochs']       = 12  # standard RT-DETR schedule
cfg['fp16']             = True

# Save overridden config
with open('configs/rtdetr_r50_coco_colab.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(yaml.dump(cfg, default_flow_style=False))

## 5 — Smoke test (inference only)

Verify the model loads and runs correctly on a sample image.

In [ ]:
import sys
sys.path.insert(0, '/content/detrflow')

from inference.predictor import RTDetrPredictor
from inference.visualizer import draw_detections
from PIL import Image
import requests
from IPython.display import display

# Load sample image
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
img = Image.open(requests.get(url, stream=True).raw)

# Run inference
predictor = RTDetrPredictor(model_id='PekingU/rtdetr_r50vd', threshold=0.5)
detections = predictor.predict(img)

print(f'Detections: {len(detections)}')
for d in detections[:5]:
    print(f"  {d['label']:20s} score={d['score']:.3f}  box={[round(x) for x in d['box']]}")

# Visualize
vis = draw_detections(img, detections)
display(vis)

## 6 — Training

> **T4 (~16 GB):** ~3-4 hrs/epoch × 12 epochs = ~36-48 hrs total.  
> **A100 (~40 GB):** ~45 min/epoch × 12 epochs = ~9 hrs total.  
> A checkpoint is saved to Drive after every epoch to guard against session disconnects.

In [ ]:
!python scripts/train.py --config configs/rtdetr_r50_coco_colab.yaml

## 6b — Resume from checkpoint

If your session disconnected, resume training from the latest checkpoint saved to Drive.

In [ ]:
import os

# Find the latest checkpoint
ckpts = sorted([
    f for f in os.listdir(CKPT_DIR)
    if f.startswith('checkpoint-epoch')
])

if ckpts:
    latest = f'{CKPT_DIR}/{ckpts[-1]}'
    print(f'Resuming from: {latest}')
    !python scripts/train.py \
        --config configs/rtdetr_r50_coco_colab.yaml \
        --resume {latest}
else:
    print('No checkpoint found. Run cell 6 first.')

## 7 — COCO mAP Evaluation

In [ ]:
import os

# Pick the latest checkpoint (last epoch)
ckpts = sorted([
    f for f in os.listdir(CKPT_DIR)
    if f.startswith('checkpoint-epoch')
])

if not ckpts:
    print('No checkpoint found. Run training first.')
else:
    best = f'{CKPT_DIR}/{ckpts[-1]}'
    print(f'Evaluating: {best}')
    !python scripts/evaluate.py \
        --model_path {best} \
        --coco_dir data/coco \
        --split val

## 8 — Push best model to Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoModelForObjectDetection, AutoImageProcessor

HF_REPO = 'umutonuryasar/rtdetr-r50vd-coco-detrflow'

model     = AutoModelForObjectDetection.from_pretrained(best)
processor = AutoImageProcessor.from_pretrained(best)

model.push_to_hub(HF_REPO)
processor.push_to_hub(HF_REPO)

print(f'Model pushed to: https://huggingface.co/{HF_REPO}')